# 01 — Feature Engineering: Tags & Labels

Builds the foundational sparse feature matrices from the raw MusicBrainz parquet exports.

**What it does:**
- Encodes album and artist tags as a weighted sparse matrix, using normalised tag counts as signal strength
- Encodes album record labels as a sparse matrix weighted by label tag counts
- Encodes album type (e.g. studio, live, compilation) as a normalised one-hot matrix
- Creates and saves the master album and artist ID index (`album_ids.pkl`, `artist_ids.pkl`) that all downstream notebooks use to keep row alignment consistent across matrices

**Inputs:** `data/mb_album.parquet`, `data/mb_artist.parquet`, `data/mb_album_tag.parquet`, `data/mb_artist_tag.parquet`, `data/mb_album_label.parquet`

**Outputs to `data/features/`:** `album_tags_matrix.npz`, `album_labels_matrix.npz`, `album_types_matrix.npz`, `artist_tags_matrix.npz`, `album_ids.pkl`, `artist_ids.pkl`

**Run before:** `02-feature-ratings.ipynb`

## Imports

Standard scientific Python stack. `scipy.sparse.csr_matrix` is the core data structure used throughout — it stores only non-zero values and their coordinates, which matters here because the feature matrices are extremely sparse (most albums have tags covering only a tiny fraction of the full tag vocabulary). `matplotlib` and `seaborn` are imported now but only used later in the visualisation cell.

In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
import seaborn as sns

## Load Data & Define the Master ID Universe

Loads the tag and label association tables, then — crucially — loads the **full album and artist universe** from the base entity tables.

The key design decision here is loading `unique_album_ids` and `unique_artist_ids` upfront from `mb_album` and `mb_artist` rather than deriving them from the tag data. This matters because not every album has tags, and not every artist has tags. If you derived your row index from the tag data alone, albums/artists without any tags would be silently absent from the matrix. By anchoring row indices to the full universe, every album and every artist gets a guaranteed row — albums with no tags simply have an all-zero row. This is essential for **row alignment**: downstream notebooks load `album_ids.pkl` and expect row `i` in every matrix to always refer to the same album, regardless of whether that album had tags, labels, or ratings.

In [ ]:
#Load the ratings dataframe from the parquet tables we created
artist_tags = pd.read_parquet('../data/mb_artist_tag.parquet')
album_tags = pd.read_parquet('../data/mb_album_tag.parquet')
album_label = pd.read_parquet('../data/mb_album_label.parquet')

# Load the complete album and artist universes up-front — these define the master
# row indices so that albums/artists with no tags are preserved as zero-rows in
# every sparse matrix rather than being silently dropped.
unique_album_ids = pd.Index(pd.read_parquet('../data/mb_album.parquet', columns=['id'])['id'].sort_values())
unique_artist_ids = pd.Index(pd.read_parquet('../data/mb_artist.parquet', columns=['id'])['id'].sort_values())

# Verify the loads
dataframes = {
    "Artist Tags": artist_tags,
    "Album Tags": album_tags,
    "Album Label": album_label
}

for name, df in dataframes.items():
    print(f"✅ {name}: {df.shape[0]:,} rows loaded.")

print(f"✅ Full album universe: {len(unique_album_ids):,} albums")
print(f"✅ Full artist universe: {len(unique_artist_ids):,} artists")

## Build Artist Tags Sparse Matrix

Constructs the artist-by-tag feature matrix in four steps:

**1. Tag frequency filter (`>= 10`):** Tags that appear on fewer than 10 artists are dropped. This removes noise — obscure tags added by one or two users carry little generalisation signal and would inflate the column dimension with near-useless features. The threshold of 10 is a practical minimum to treat a tag as a community-level signal rather than a personal annotation.

**2. Per-artist weight normalisation:** Each artist's raw tag counts are divided by that artist's total tag count, producing a weight between 0.0 and 1.0. This means the matrix encodes the *relative* genre/style profile of an artist rather than the absolute volume of tagging activity — a heavily-tagged artist and a lightly-tagged artist become comparable.

**3. COO-style coordinate generation:** The matrix is built using the COOrdinate (COO) format concept: three parallel arrays of `(row, col, value)` triples, where each triple represents one non-zero entry. `tag_code` gives the column index (dense sequential integer for each surviving tag), and `artist_code` maps each artist_id to its position in `unique_artist_ids` using `get_indexer` — if an artist_id is not found in the universe, `get_indexer` returns -1 and that row is excluded.

**4. CSR matrix construction:** `csr_matrix((data, (rows, cols)), shape=...)` accepts the COO arrays directly and converts to Compressed Sparse Row format internally. CSR is chosen over COO for the final structure because it supports efficient row-slicing and matrix-vector multiplication — the operations used during similarity search in downstream notebooks. The explicit `shape` parameter is what guarantees the full artist universe is represented even if some artists have no tag rows in the filtered data.

In [ ]:
print("1. Filtering out rare artist tags...")
# Group by tag to count occurrences and filter out tags appearing < 10 times
artist_tag_counts = artist_tags.groupby('tag_id').size()
popular_artist_tags = artist_tag_counts[artist_tag_counts >= 10].index
artist_tags_filtered = artist_tags[artist_tags['tag_id'].isin(popular_artist_tags)].copy()

print("2. Normalizing artist tag weights...")
# Calculate relative weights so an artist's profile bounds between 0.0 and 1.0
artist_totals = artist_tags_filtered.groupby('artist_id')['tag_count'].transform('sum')
artist_tags_filtered['tag_weight'] = (artist_tags_filtered['tag_count'] / artist_totals).astype('float32')

print("3. Generating category codes...")
# Tags get dense sequential codes within the filtered set
artist_tags_filtered['tag_code'] = artist_tags_filtered['tag_id'].astype('category').cat.codes
unique_artist_tag_ids = artist_tags_filtered['tag_id'].astype('category').cat.categories

# Artist rows are mapped against the FULL artist universe (unique_artist_ids, loaded above)
# so artists without popular tags still occupy a row as zeros — no artists dropped
artist_tags_filtered['artist_code'] = unique_artist_ids.get_indexer(artist_tags_filtered['artist_id'])

print("4. Building Artist Sparse Matrix via COOrdinate mapping...")
# Extract coordinate arrays
artist_row_indices = artist_tags_filtered['artist_code'].values
artist_col_indices = artist_tags_filtered['tag_code'].values
artist_weights = artist_tags_filtered['tag_weight'].values

# Construct the sparse matrix directly
X_artist_tags_sparse = csr_matrix(
    (artist_weights, (artist_row_indices, artist_col_indices)), 
    shape=(len(unique_artist_ids), len(unique_artist_tag_ids))
)

print(f"🚀 Done! Artist Matrix Shape: {X_artist_tags_sparse.shape}")
print(f"Non-zero elements tracked: {X_artist_tags_sparse.nnz}")

## Build Album Tags Sparse Matrix

Identical pipeline to the artist tags cell above, applied to albums instead. The same four steps apply: frequency filter at `>= 10`, per-album weight normalisation, coordinate code generation, and CSR construction.

One point worth noting: `unique_tag_ids` (the album tag vocabulary) and `unique_artist_tag_ids` (from the cell above) are separate objects and will generally have different lengths — album tags and artist tags come from independent tagging activity in MusicBrainz and are not required to be the same vocabulary. Downstream notebooks treat the album tags matrix and artist tags matrix as independent feature sources.

The `shape=(len(unique_album_ids), len(unique_tag_ids))` argument is again what anchors this matrix to the full album universe, ensuring row alignment with every other album matrix built in this notebook.

In [ ]:
print("1. Filtering out rare tags...")
# Group by tag to count occurrences and filter rare ones (>= 10)
tag_counts = album_tags.groupby('tag_id').size()
popular_tags = tag_counts[tag_counts >= 10].index
album_tags_filtered = album_tags[album_tags['tag_id'].isin(popular_tags)].copy()

print("2. Normalizing tag weights...")
# Calculate relative weights using transform('sum') on the filtered set
album_totals = album_tags_filtered.groupby('album_id')['tag_count'].transform('sum')
album_tags_filtered['tag_weight'] = (album_tags_filtered['tag_count'] / album_totals).astype('float32')

print("3. Generating category codes...")
# Tags get dense sequential codes within the filtered set
album_tags_filtered['tag_code'] = album_tags_filtered['tag_id'].astype('category').cat.codes
unique_tag_ids = album_tags_filtered['tag_id'].astype('category').cat.categories

# Album rows are mapped against the FULL album universe (unique_album_ids, loaded above)
# so albums without popular tags still occupy a row as zeros — no albums dropped
album_tags_filtered['album_code'] = unique_album_ids.get_indexer(album_tags_filtered['album_id'])

print("4. Building Sparse Matrix instantly via COOrdinate mapping...")
# Extract our rows, columns, and data values as raw numpy arrays
row_indices = album_tags_filtered['album_code'].values
col_indices = album_tags_filtered['tag_code'].values
weights = album_tags_filtered['tag_weight'].values

# Create the CSR sparse matrix directly from the coordinates
# Dimensions: Number of unique albums x Number of unique popular tags
X_album_tags_sparse = csr_matrix(
    (weights, (row_indices, col_indices)), 
    shape=(len(unique_album_ids), len(unique_tag_ids))
)

print(f"🚀 Done! Matrix Shape: {X_album_tags_sparse.shape}")
print(f"Non-zero data elements tracked: {X_album_tags_sparse.nnz}")

## Build Album Labels & Types Sparse Matrices

Constructs two further album-level feature matrices from the `mb_album_label` table.

**Label matrix:** Record labels are treated similarly to tags — rare labels (fewer than 10 albums) are dropped, surviving labels are normalised by the album's total label-tag count, and a CSR matrix is built with the same COO approach. The weight here reflects how strongly a label is associated with an album relative to that album's other label relationships.

**Row alignment enforcement:** Instead of using `get_indexer` (as in the tag cells), album_id is cast to a `pd.Categorical` with `categories=unique_album_ids`. This enforces the exact same row ordering established upfront. Any album_id in the label data that does not exist in `unique_album_ids` will become NaN after the cast and is dropped via `dropna`. This is intentional — it means only albums that exist in the master universe appear in the matrix, and their row positions are guaranteed to match.

**Types matrix:** `label_type` is a MusicBrainz-coded integer indicating the kind of label (e.g. imprint, original production, bootleg). The type matrix is built as a binary indicator (all values 1.0) then **L1-normalised row-wise** using scikit-learn's `normalize`. L1 normalisation divides each row by its row sum, so a row with one label type gets value 1.0 and a row with two label types gets 0.5 each. This keeps all values in [0, 1] and makes rows with different numbers of label relationships comparable in magnitude — without it, albums with many labels would have artificially larger feature vectors than albums with few.

In [ ]:
print("1. Filtering out rare labels...")
album_label['label_type'] = album_label['label_type'].fillna(0.0).astype('int32')
label_counts = album_label.groupby('label_id').size()
popular_labels = label_counts[label_counts >= 10].index
label_filtered = album_label[album_label['label_id'].isin(popular_labels)].copy()

# Deduplicate to one row per (album, label) pair before weighting.
# The parquet has one row per (album, label, tag) triple, so tag_count-based
# weighting would unfairly penalise labels with fewer tags. Equal weight per
# label regardless of how heavily tagged it is.
label_filtered = label_filtered.drop_duplicates(subset=['album_id', 'label_id'])

print("2. Normalizing label weights...")
label_totals = label_filtered.groupby('album_id')['label_id'].transform('count')
label_filtered['label_weight'] = (1.0 / label_totals).astype('float32')

print("3. Generating category codes...")
# CRUCIAL: We map album_id based on the exact same universe of unique_album_ids from your tags step
# This guarantees that Row 5 in this matrix is the exact same album as Row 5 in your tag matrix!
label_filtered['album_id'] = pd.Categorical(label_filtered['album_id'], categories=unique_album_ids)
label_filtered = label_filtered.dropna(subset=['album_id']).copy() # Drop any albums that didn't have tags

label_filtered['album_code'] = label_filtered['album_id'].cat.codes
label_filtered['label_code'] = label_filtered['label_id'].astype('category').cat.codes
label_filtered['type_code'] = label_filtered['label_type'].astype('category').cat.codes

unique_label_ids = label_filtered['label_id'].astype('category').cat.categories
unique_type_ids = label_filtered['label_type'].astype('category').cat.categories

print("4. Building Sparse Matrices...")
# Label Matrix
X_album_labels_sparse = csr_matrix(
    (label_filtered['label_weight'].values, (label_filtered['album_code'].values, label_filtered['label_code'].values)),
    shape=(len(unique_album_ids), len(unique_label_ids))
)

# Label Type Matrix (Binary indicator)
# We give it a uniform weight of 1.0, then normalize the rows so they scale between 0 and 1
ones = np.ones(len(label_filtered), dtype='float32')
X_album_types_sparse = csr_matrix(
    (ones, (label_filtered['album_code'].values, label_filtered['type_code'].values)),
    shape=(len(unique_album_ids), len(unique_type_ids))
)
# Normalize row-wise so multiple labels per album don't push values past 1.0
from sklearn.preprocessing import normalize
X_album_types_sparse = normalize(X_album_types_sparse, norm='l1', axis=1)

print(f"🚀 Labels Shape: {X_album_labels_sparse.shape} | Types Shape: {X_album_types_sparse.shape}")

## Visualise Sparse Matrix Structure

Diagnostic plots to validate the structural properties of the three matrices before saving. Each matrix gets two plots:

- **Profile complexity (left column):** A histogram of non-zero entries per row (`getnnz(axis=1)`). This shows how many features the typical album or artist has — a heavy concentration near zero would indicate the filtering thresholds are too aggressive and most entities are left with empty profiles.

- **Long-tail feature popularity (right column):** A sorted curve of non-zero entries per column (`getnnz(axis=0)`) on a log scale. This reveals the power-law distribution that tag/label data always exhibits — a few features (e.g. "rock", "pop") appear on a huge number of albums, while most features appear on only a handful. The log scale makes the tail visible. This is useful to confirm the `>= 10` filter has removed the very bottom of the tail without cutting off the mid-range.

The figure is saved to `sparse_features_structural_analysis.png` in the notebook directory for reference.

In [ ]:
# Set up a clean, professional aesthetic for the report
sns.set_theme(style="whitegrid")

# Create a spacious 3x2 grid of subplots
fig, axes = plt.subplots(3, 2, figsize=(16, 18))

# ----------------------------------------------------------------------
# ROW 1: ALBUM TAGS MATRIX VISUALIZATION
# ----------------------------------------------------------------------
# Left: Profile Complexity (Non-zero tags per album)
album_tags_per_row = X_album_tags_sparse.getnnz(axis=1)
sns.histplot(album_tags_per_row, bins=range(0, 35), ax=axes[0, 0], color='#4A90E2', kde=True)
axes[0, 0].set_title('Album Tags: Profile Complexity (Tags per Album)', fontsize=12, weight='bold')
axes[0, 0].set_xlabel('Number of Unique Tags on a Single Album')
axes[0, 0].set_ylabel('Count of Albums')
axes[0, 0].set_xlim(0, 30)

# Right: Long-Tail Distribution (Album Tag Popularity)
album_tag_popularity = np.sort(X_album_tags_sparse.getnnz(axis=0))[::-1]
axes[0, 1].plot(album_tag_popularity, color='#4A90E2', linewidth=2.5)
axes[0, 1].fill_between(range(len(album_tag_popularity)), album_tag_popularity, color='#4A90E2', alpha=0.25)
axes[0, 1].set_yscale('log')
axes[0, 1].set_title('Album Tags: Long-Tail Feature Popularity', fontsize=12, weight='bold')
axes[0, 1].set_xlabel('Tag Index (Sorted by Global Popularity)')
axes[0, 1].set_ylabel('Number of Albums Sharing Tag (Log Scale)')


# ----------------------------------------------------------------------
# ROW 2: ALBUM LABELS MATRIX VISUALIZATION
# ----------------------------------------------------------------------
# Left: Profile Complexity (Non-zero labels per album)
album_labels_per_row = X_album_labels_sparse.getnnz(axis=1)
sns.histplot(album_labels_per_row, bins=range(0, 10), ax=axes[1, 0], color='#E056FD', kde=False)
axes[1, 0].set_title('Album Labels: Profile Complexity (Labels per Album)', fontsize=12, weight='bold')
axes[1, 0].set_xlabel('Number of Unique Record Labels on a Single Album')
axes[1, 0].set_ylabel('Count of Albums')
axes[1, 0].set_xlim(0, 6)

# Right: Long-Tail Distribution (Album Label Popularity)
album_label_popularity = np.sort(X_album_labels_sparse.getnnz(axis=0))[::-1]
axes[1, 1].plot(album_label_popularity, color='#E056FD', linewidth=2.5)
axes[1, 1].fill_between(range(len(album_label_popularity)), album_label_popularity, color='#E056FD', alpha=0.25)
axes[1, 1].set_yscale('log')
axes[1, 1].set_title('Album Labels: Long-Tail Feature Popularity', fontsize=12, weight='bold')
axes[1, 1].set_xlabel('Label Index (Sorted by Global Popularity)')
axes[1, 1].set_ylabel('Number of Albums Sharing Label (Log Scale)')


# ----------------------------------------------------------------------
# ROW 3: ARTIST TAGS MATRIX VISUALIZATION
# ----------------------------------------------------------------------
# Left: Profile Complexity (Non-zero tags per artist)
artist_tags_per_row = X_artist_tags_sparse.getnnz(axis=1)
sns.histplot(artist_tags_per_row, bins=range(0, 45), ax=axes[2, 0], color='#10AC84', kde=True)
axes[2, 0].set_title('Artist Tags: Profile Complexity (Tags per Artist)', fontsize=12, weight='bold')
axes[2, 0].set_xlabel('Number of Unique Tags on a Single Artist')
axes[2, 0].set_ylabel('Count of Artists')
axes[2, 0].set_xlim(0, 40)

# Right: Long-Tail Distribution (Artist Tag Popularity)
artist_tag_popularity = np.sort(X_artist_tags_sparse.getnnz(axis=0))[::-1]
axes[2, 1].plot(artist_tag_popularity, color='#10AC84', linewidth=2.5)
axes[2, 1].fill_between(range(len(artist_tag_popularity)), artist_tag_popularity, color='#10AC84', alpha=0.25)
axes[2, 1].set_yscale('log')
axes[2, 1].set_title('Artist Tags: Long-Tail Feature Popularity', fontsize=12, weight='bold')
axes[2, 1].set_xlabel('Tag Index (Sorted by Global Popularity)')
axes[2, 1].set_ylabel('Number of Artists Sharing Tag (Log Scale)')


# Adjust subplots to ensure labels and titles are clearly readable and un-truncated
plt.tight_layout()

# Save the full visual matrix directly to your repository folder
plt.savefig('sparse_features_structural_analysis.png', dpi=300)

## Save ID Mappings (Intermediate Checkpoint)

Saves `album_ids.pkl` and `artist_ids.pkl` to `../data/features/` as plain Python lists.

These files are the **row index contract** between this notebook and all downstream notebooks. Any notebook that loads a sparse matrix must also load the corresponding ID file to know which entity corresponds to which row. For example, to look up the feature vector for a specific album, a downstream notebook does: `row = album_ids.index(target_album_id)` then `matrix[row]`.

This cell is an intermediate save — it saves the ID mappings before the matrices are ready, which allows other development work to proceed against the index without running the full matrix build. The final save cell below overwrites these files along with saving the matrices.

In [ ]:
import os
import pickle

os.makedirs('../data/features', exist_ok=True)

# Save your ID mappings so other notebooks can read the row alignments
with open('../data/features/artist_ids.pkl', 'wb') as f:
    pickle.dump(unique_artist_ids.tolist(), f)

with open('../data/features/album_ids.pkl', 'wb') as f:
    pickle.dump(unique_album_ids.tolist(), f)

## Save All Sparse Matrices & ID Mappings (Final Output)

Persists all four sparse matrices as `.npz` files and overwrites the ID mapping pickles with final versions.

**`.npz` format:** `scipy.sparse.save_npz` serialises a CSR matrix into NumPy's compressed archive format, storing the sparse data arrays (`data`, `indices`, `indptr`) rather than a dense representation. This keeps file sizes small and allows `load_npz` in downstream notebooks to reconstruct the exact CSR matrix without any reprocessing.

**Why overwrite the pickles again:** The ID lists saved here are identical to those saved in the previous cell. The duplication is intentional — this cell is the canonical single entry point for a clean full run of the notebook, and keeping the matrix saves and ID saves together means you can always re-run just this cell (after the matrices are built in memory) to refresh all outputs atomically.

After this cell runs, `../data/features/` contains everything downstream notebooks need: four `.npz` matrix files and two `.pkl` index files. No other output from this notebook is required downstream.

In [ ]:
import os
import pickle
from scipy.sparse import save_npz, hstack

os.makedirs('../data/features', exist_ok=True)

# Combine label identity and label type into a single record_label matrix.
# Both encode the same underlying fact (which label released the album)
# from two angles — identity (3,560 cols) and type (10 cols).
X_record_label = hstack([X_album_labels_sparse, X_album_types_sparse]).tocsr()

print("Saving Sparse Matrices to ../data/features/...")
save_npz('../data/features/artist_tags_matrix.npz', X_artist_tags_sparse)
save_npz('../data/features/album_tags_matrix.npz', X_album_tags_sparse)
save_npz('../data/features/album_record_label_matrix.npz', X_record_label)

print("Saving Index IDs to ../data/features/...")
with open('../data/features/artist_ids.pkl', 'wb') as f:
    pickle.dump(unique_artist_ids.tolist(), f)

with open('../data/features/album_ids.pkl', 'wb') as f:
    pickle.dump(unique_album_ids.tolist(), f)

print(f'record_label matrix: {X_record_label.shape}  nnz={X_record_label.nnz:,}')
print("🎉 Metadata successfully exported to ../data/features/")
